In [ ]:
import pandas as pd
import zarr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from sample_db import SampleDB

db = SampleDB()

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
records = []
for b in range(capture_buffers_zarr.nchunks):
    sample = capture_buffers_zarr.blocks[b]
    ch0_of_sample = sample[:, 0]
    stats = calculate_audio_stats(ch0_of_sample, ignore_in_out=500)
    records.append(stats)
print("capture_buffers_zarr ch0")
capture_buffers_df = pd.DataFrame(records)
print(capture_buffers_df.describe())

In [ ]:
capture_buffers_df.head()

In [ ]:
corr_matrix = capture_buffers_df.corr()
plt.figure(figsize=(8, 6))
ax = sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, annot=False)
for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):
        ax.text(j + 0.5, i + 0.5, f"{corr_matrix.iloc[i, j]:.2f}",
                ha='center', va='center', color='black', fontsize=10)
plt.show()

### distributiuon of losses from score

In [ ]:
losses = db.losses_for(run='004', model='230_keras/i0')
losses_1_df = pd.DataFrame(losses)
losses_1_df['src'] = 'sobol'
losses_1_df.describe()

In [ ]:
losses = []
for run in range(200, 210):
    run = f"{run:03d}"
    losses.extend(db.losses_for(run=run, model='230_keras/i0'))
losses_2_df = pd.DataFrame(losses)
losses_2_df['src'] = 'hard_mined'
losses_2_df.describe()

In [ ]:
losses_df = pd.concat([losses_1_df, losses_2_df], ignore_index=True)
losses_df.tail()

In [ ]:
src_values = losses_df['src'].unique()
cols = ['huber', 'stft']

fig, axes = plt.subplots(len(src_values), len(cols), figsize=(12, 4 * len(src_values)), sharex='col')

for row, src in enumerate(src_values):
    subset = losses_df[losses_df['src'] == src]
    for col_idx, col in enumerate(cols):
        ax = axes[row, col_idx]
        sns.histplot(data=subset, x=col, kde=True, stat="density", alpha=0.5, ax=ax)
        ax.set_title(f"{col} — {src}")

plt.tight_layout()
plt.show()
